# SDL — code registry manager (edit / add your codes)

The list of scientific **codes** offered by the manifest builders (and the keywords / theme applied at
publish time) lives in an **editable file** `sdl_codes.yml`, kept next to `sdl_lib.py`. This notebook
lets you **view, add, edit and delete** codes there **without touching any Python** — so you keep full
control of what is published under each of your codes.

`sdl_lib` loads `sdl_codes.yml` and merges it over its built-in defaults (search order:
`$SDL_CODES_FILE` → next to `sdl_lib.py` → current folder). No login is needed: this only edits a local
file.

**Flow:** *Setup → manage codes*. Pick `(new)` to add one, or pick an existing id to edit it; **Save**
writes the file. Afterwards, **re-run the Setup cell** in the builder notebooks (it reloads `sdl_lib`)
so the new code appears in the *Code* selector.


## 1. Setup & dependencies

In [ ]:
# --- Setup: load the companion library sdl_lib.py and check dependencies -----
import os, sys, importlib, importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Locate sdl_lib.py: it lives either in a sibling 01_sdl_publish/ folder (repo
# layout) or directly in the parent folder (server ~/SDL layout). Search both.
_cands = [os.path.join(os.getcwd(), "..", "01_sdl_publish"),
          os.path.join(os.getcwd(), ".."),
          os.getcwd(),
          os.path.expanduser("~/HySEALab/Notebooks/01_sdl_publish"),  # HySEALab menu: shared lib
          os.path.expanduser("~/HySEALab/Notebooks"),                 # shared (next to map viewer)
          os.path.expanduser("~/SDL")]                                # personal working copy
LIB_DIR = next((os.path.abspath(d) for d in _cands
                if os.path.exists(os.path.join(d, "sdl_lib.py"))), None)
if LIB_DIR is None:
    raise FileNotFoundError("sdl_lib.py not found in ../01_sdl_publish, .. or .")
sys.path.insert(0, LIB_DIR)

REQUIRED = {
    "requests": "requests", "requests_toolbelt": "requests-toolbelt",
    "rasterio": "rasterio", "cmocean": "cmocean",
    "matplotlib": "matplotlib", "numpy": "numpy", "tqdm": "tqdm",
    "yaml": "pyyaml", "jsonschema": "jsonschema",
}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

_dep_out = widgets.Output()
def _install_missing(_):
    import subprocess
    with _dep_out:
        clear_output()
        print("Installing into the user site:", " ".join(_missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", *_missing])
        print("Done. Restart the kernel and re-run this cell.")
_btn_install = widgets.Button(description="Install missing deps", button_style="warning")
_btn_install.on_click(_install_missing)

if _missing:
    print("Missing dependencies:", ", ".join(_missing))
    display(_btn_install, _dep_out)
else:
    print("All dependencies available.")

import sdl_lib
importlib.reload(sdl_lib)
print("sdl_lib loaded from:", LIB_DIR)

# Shared state for this notebook (login token lives only in memory).
S = {"base_url": "https://sdl-dev.hpc.cineca.it", "token": None, "headers": None}
print("Setup ready.")


## 2. View / add / edit / delete codes

Each code has: **Code id** (what goes into `experiment.code.name`), **Display** name, **Keywords**
(merged with the experiment's), **Theme** (SDL theme used when creating the experiment) and an optional
**Description** prefix.

In [ ]:
# --- Manage the editable code registry (sdl_codes.yml next to sdl_lib.py) -----
import yaml, copy
CODES_FILE = os.path.join(LIB_DIR, "sdl_codes.yml")

def _load_reg():
    if os.path.exists(CODES_FILE):
        try:
            with open(CODES_FILE, encoding="utf-8") as f:
                return yaml.safe_load(f) or {}
        except Exception as e:
            print("Could not read", CODES_FILE, "->", e)
    return copy.deepcopy(getattr(sdl_lib, "CODE_METADATA_REGISTRY", {}))  # fall back to built-ins

REG = _load_reg()

def _show():
    print(f"Codes in {CODES_FILE}:")
    print("-" * 70)
    if not REG:
        print("(empty)")
    for k, v in REG.items():
        print(f"- {k}  ->  {v.get('display_name','')}")
        print(f"    keywords: {', '.join(v.get('keywords', []))}")
        print(f"    theme   : {', '.join(v.get('default_theme', []))}")
    print("-" * 70)

m_pick = widgets.Dropdown(options=["(new)"] + list(REG), value="(new)", description="Edit:",
                          layout=widgets.Layout(width="460px"))
m_key = widgets.Text(description="Code id:", placeholder="e.g. multilayer_hysea",
                     layout=widgets.Layout(width="460px"))
m_name = widgets.Text(description="Display:", placeholder="e.g. Multilayer-HySEA",
                      layout=widgets.Layout(width="460px"))
m_kw = widgets.Text(description="Keywords:", placeholder="comma-separated",
                    layout=widgets.Layout(width="660px"))
m_theme = widgets.Text(description="Theme:", value="volcanoobservatis",
                       placeholder="comma-separated", layout=widgets.Layout(width="460px"))
m_desc = widgets.Textarea(description="Descr.:", layout=widgets.Layout(width="660px", height="60px"))
m_save = widgets.Button(description="Save code", button_style="success")
m_del = widgets.Button(description="Delete code", button_style="danger")
m_out = widgets.Output()

def _split(s):
    return [x.strip() for x in s.replace(";", ",").split(",") if x.strip()]

def _on_pick(_=None):
    k = m_pick.value
    if k == "(new)":
        m_key.value = m_name.value = m_kw.value = m_desc.value = ""
        m_theme.value = "volcanoobservatis"
        return
    v = REG.get(k, {})
    m_key.value = k
    m_name.value = v.get("display_name", "")
    m_kw.value = ", ".join(v.get("keywords", []))
    m_theme.value = ", ".join(v.get("default_theme", [])) or "volcanoobservatis"
    m_desc.value = v.get("description_prefix", "")
m_pick.observe(_on_pick, names="value")

def _persist():
    with open(CODES_FILE, "w", encoding="utf-8") as f:
        yaml.safe_dump(REG, f, sort_keys=False, allow_unicode=True)

def _save(_):
    with m_out:
        clear_output()
        key = m_key.value.strip().lower()
        if not key:
            print("Code id is required."); return
        if not m_name.value.strip():
            print("Display name is required."); return
        REG[key] = {"display_name": m_name.value.strip(),
                    "keywords": _split(m_kw.value),
                    "description_prefix": m_desc.value.strip(),
                    "default_theme": _split(m_theme.value) or ["volcanoobservatis"]}
        _persist()
        m_pick.options = ["(new)"] + list(REG)
        print(f"Saved '{key}'. Wrote {CODES_FILE}.")
        print("Re-run the Setup cell of the builder notebooks (it reloads sdl_lib) to pick it up.")
        _show()
m_save.on_click(_save)

def _delete(_):
    with m_out:
        clear_output()
        key = m_key.value.strip().lower()
        if key not in REG:
            print(f"'{key}' is not in the registry."); return
        REG.pop(key)
        _persist()
        m_pick.options = ["(new)"] + list(REG)
        m_pick.value = "(new)"
        print(f"Deleted '{key}'. Wrote {CODES_FILE}.")
        _show()
m_del.on_click(_delete)

with m_out:
    _show()
display(widgets.VBox([m_pick, m_key, m_name, m_kw, m_theme, m_desc,
                      widgets.HBox([m_save, m_del]), m_out]))


## Notes

- **Where it is saved:** `sdl_codes.yml` next to `sdl_lib.py` (on the server, `~/SDL/sdl_codes.yml`).
  You can also edit that YAML by hand.
- **Propagation:** the builders read the registry at *Setup* time; after editing here, re-run their
  Setup cell (it `importlib.reload`s `sdl_lib`, which re-reads the file). The headless
  `sdl_publish_manifest.py` also picks it up on its next run.
- **Theme:** kept as `volcanoobservatis` by default because that value is accepted by the API when
  creating an experiment; change it if CINECA gives you a tsunami-specific theme id.
- **No effect on already-published experiments** — the registry only influences *new* publications.
